# Auto-MQT Local Smoke Test (3050 Ti)

Use this notebook to validate the full pipeline locally with a lightweight VQA backend before using Colab GPU credits.

This smoke backend uses `Salesforce/blip-vqa-base` and ignores `visual_tokens`. It is for pipeline correctness checks, not final Auto-MQT metrics.

In [1]:
from pathlib import Path
import os
import sys
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

print('root:', ROOT)
print('cuda_available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

root: C:\GitHub_Repos\Auto-MQT-
cuda_available: True
device: NVIDIA GeForce RTX 3050 Ti Laptop GPU


## 1) Configure Lightweight Local Backend

In [2]:
os.environ['MQT_LLAVA_BACKEND'] = 'blip_vqa_smoke'
os.environ['AUTO_MQT_SMOKE_MODEL'] = 'Salesforce/blip-vqa-base'
os.environ['AUTO_MQT_SMOKE_DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['AUTO_MQT_SMOKE_MAX_NEW_TOKENS'] = '20'

for key in ['MQT_LLAVA_BACKEND', 'AUTO_MQT_SMOKE_MODEL', 'AUTO_MQT_SMOKE_DEVICE']:
    print(key, '=', os.environ[key])

MQT_LLAVA_BACKEND = blip_vqa_smoke
AUTO_MQT_SMOKE_MODEL = Salesforce/blip-vqa-base
AUTO_MQT_SMOKE_DEVICE = cuda


## 2) Build Tiny Dataset Manifests

In [3]:
%cd {ROOT}
!python src/prepare_datasets.py --config configs/datasets.yaml --datasets textvqa --train-limit 8 --eval-limit 8 --prompt-style none
!python src/verify_manifest.py --manifest data/manifests/train.jsonl
!python src/verify_manifest.py --manifest data/manifests/eval.jsonl

C:\GitHub_Repos\Auto-MQT-
textvqa:train -> kept=8 skipped=0 limit=8
textvqa:train -> kept=8 skipped=0 limit=8
wrote train rows: 8
wrote eval rows: 8


c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ovalt\.cache\huggingface\hub\datasets--fedlib--TextVQA-Data. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to reg

manifest ok: data/manifests/train.jsonl
rows: 8
manifest ok: data/manifests/eval.jsonl
rows: 8


## 3) Tiny Baseline Run

In [4]:
%cd {ROOT}
!python src/evaluate_token_policy.py --data data/manifests/eval.jsonl --fixed-budget 36 --prompt-style short --out results/smoke_fixed_36.jsonl

C:\GitHub_Repos\Auto-MQT-
examples: 8
exact_accuracy: 0.1250
relaxed_accuracy: 0.1250
avg_visual_tokens: 36.00
avg_latency_s: 0.4522


c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
W0522 16:05:21.975000 93732 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Use

## 4) Tiny Oracle Labels

In [5]:
%cd {ROOT}
!python src/oracle_labeling.py --data data/manifests/train.jsonl --out data/manifests/smoke_oracle_train.jsonl --budgets 16 36 --prompt-style short --score-key relaxed_match --limit 6

C:\GitHub_Repos\Auto-MQT-
label example 1: example_id=textvqa_train_3
wrote example_id=textvqa_train_3 oracle_budget=36 oracle_score=0.0
label example 2: example_id=textvqa_train_303
wrote example_id=textvqa_train_303 oracle_budget=36 oracle_score=0.0
label example 3: example_id=textvqa_train_461
wrote example_id=textvqa_train_461 oracle_budget=36 oracle_score=0.0
label example 4: example_id=textvqa_train_1882
wrote example_id=textvqa_train_1882 oracle_budget=36 oracle_score=0.0
label example 5: example_id=textvqa_train_2059
wrote example_id=textvqa_train_2059 oracle_budget=36 oracle_score=0.0
label example 6: example_id=textvqa_train_2890
wrote example_id=textvqa_train_2890 oracle_budget=36 oracle_score=0.0
new_examples: 6
avg_new_oracle_budget: 36.00
output: data/manifests/smoke_oracle_train.jsonl


c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
W0522 16:08:44.227000 72000 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Use

## 5) Extract Frozen Features (Small CLIP)

In [6]:
%cd {ROOT}
!python src/extract_router_features.py --data data/manifests/smoke_oracle_train.jsonl --out data/manifests/smoke_oracle_train_with_features.jsonl --clip-model openai/clip-vit-base-patch32 --batch-size 8 --normalize
!python src/extract_router_features.py --data data/manifests/eval.jsonl --out data/manifests/smoke_eval_with_features.jsonl --clip-model openai/clip-vit-base-patch32 --batch-size 8 --normalize

C:\GitHub_Repos\Auto-MQT-
rows_total: 6
rows_pending: 6
device: cuda
dtype: torch.float16
clip_model: openai/clip-vit-base-patch32
wrote_rows: 6
output: data/manifests/smoke_oracle_train_with_features.jsonl


c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
W0522 16:10:09.757000 45140 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Use

rows_total: 8
rows_pending: 8
device: cuda
dtype: torch.float16
clip_model: openai/clip-vit-base-patch32
wrote_rows: 8
output: data/manifests/smoke_eval_with_features.jsonl


c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
W0522 16:11:30.605000 42536 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Use

## 6) Quick Router Config

In [7]:
import yaml

smoke_config = {
    'budgets': [16, 36],
    'prompt_dim': 512,
    'image_dim': 512,
    'hidden_dim': 128,
    'dropout': 0.1,
    'learning_rate': 0.001,
    'weight_decay': 0.0001,
    'batch_size': 8,
    'epochs': 5,
    'lambda_cost': 0.05,
    'cost_power': 1.0,
    'confidence_fallback_threshold': 0.55,
}

smoke_config_path = ROOT / 'configs' / 'router_smoke.yaml'
with smoke_config_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(smoke_config, f, sort_keys=False)

print('wrote:', smoke_config_path)

wrote: C:\GitHub_Repos\Auto-MQT-\configs\router_smoke.yaml


## 7) Train + Evaluate Router

In [8]:
%cd {ROOT}
!python src/train_router.py --labels data/manifests/smoke_oracle_train_with_features.jsonl --config configs/router_smoke.yaml --mode multimodal --out checkpoints/router_smoke_multimodal.pt
!python src/evaluate_router.py --data data/manifests/smoke_eval_with_features.jsonl --checkpoint checkpoints/router_smoke_multimodal.pt --out results/smoke_router_multimodal.jsonl

C:\GitHub_Repos\Auto-MQT-
epoch=01 loss=0.6851 val_acc=1.0000 avg_budget=36.00 regret=0.00 under=0.0000
epoch=02 loss=0.6735 val_acc=1.0000 avg_budget=36.00 regret=0.00 under=0.0000
epoch=03 loss=0.6557 val_acc=1.0000 avg_budget=36.00 regret=0.00 under=0.0000
epoch=04 loss=0.6417 val_acc=1.0000 avg_budget=36.00 regret=0.00 under=0.0000
epoch=05 loss=0.6217 val_acc=1.0000 avg_budget=36.00 regret=0.00 under=0.0000
saved: checkpoints\router_smoke_multimodal.pt
examples: 8
exact_accuracy: 0.1250
relaxed_accuracy: 0.1250
avg_visual_tokens: 36.00
avg_latency_s: 0.2442


c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
W0522 16:11:41.537000 33916 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\Ovalt\miniconda3\envs\ecs271_auto_mqt\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Use

## 8) Summarize

In [9]:
%cd {ROOT}
!python src/analyze_results.py --inputs smoke_fixed_36=results/smoke_fixed_36.jsonl smoke_router=results/smoke_router_multimodal.jsonl --out-csv results/smoke_summary.csv

C:\GitHub_Repos\Auto-MQT-
run            | examples | exact  | relaxed | avg_tokens | avg_latency_s | regret | under_rate
---------------+----------+--------+---------+------------+---------------+--------+-----------
smoke_fixed_36 | 8        | 0.1250 | 0.1250  | 36.00      | 0.452         | nan    | nan       
smoke_router   | 8        | 0.1250 | 0.1250  | 36.00      | 0.244         | nan    | nan       
wrote_csv: results\smoke_summary.csv
